## Clone YOLOv5 Repo v6.2 (w/out ultralytics) + Install Dependencies

In [1]:
# ✅ Clone YOLOv5 and lock to v6.2
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!git checkout v6.2

# ✅ Install YOLOv5 dependencies
!pip install -r requirements.txt
!pip install --upgrade protobuf==4.21.12

# ✅ Patch train.py to fix PyTorch 2.6+ pickle unpickling error in two places:
# (1) for model loading
# (2) for model saving (strip_optimizer)
!sed -i "s/torch.load(weights, map_location='cpu')/torch.load(weights, map_location='cpu', weights_only=False)/" train.py
!sed -i "s/torch.load(f, map_location=torch.device('cpu'))/torch.load(f, map_location=torch.device('cpu'), weights_only=False)/" utils/general.py

Cloning into 'yolov5'...
remote: Enumerating objects: 17488, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 17488 (delta 0), reused 0 (delta 0), pack-reused 17486 (from 2)
Receiving objects: 100% (17488/17488), 16.55 MiB | 27.92 MiB/s, done.
Resolving deltas: 100% (11995/11995), done.
/kaggle/working/yolov5
Note: switching to 'v6.2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at d3ea0df8 New YOLOv5 Classification Models (#8956)
   ━━━━━━━━━━━━━━━

## Prepare Dataset (Mount & Copy Dataset)
Go to the Kaggle Face Mask Detection dataset page: https://www.kaggle.com/datasets/andrewmvd/face-mask-detection

Add it to your notebook using “Add Dataset” (right panel in Kaggle editor).

In [2]:
# Create project dataset folder
!mkdir -p ../face-mask-data/images ../face-mask-data/labels

# Move images and annotations to working folder
!cp -r /kaggle/input/face-mask-detection/images/* ../face-mask-data/images
!cp -r /kaggle/input/face-mask-detection/annotations ../face-mask-data/annotations

## Convert Annotations to YOLO Format

In [3]:
import xml.etree.ElementTree as ET
from pathlib import Path
import os

# Define class mapping
classes = ['with_mask', 'mask_weared_incorrect', 'without_mask']
input_ann = Path('../face-mask-data/annotations')
output_labels = Path('../face-mask-data/labels')
output_labels.mkdir(parents=True, exist_ok=True)

def convert_xml_to_yolo(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    img_w = int(root.find('size/width').text)
    img_h = int(root.find('size/height').text)

    lines = []
    for obj in root.iter('object'):
        cls = obj.find('name').text
        if cls not in classes:
            continue
        cls_id = classes.index(cls)
        xmlbox = obj.find('bndbox')
        b = [float(xmlbox.find(tag).text) for tag in ('xmin', 'ymin', 'xmax', 'ymax')]
        x_center = ((b[0] + b[2]) / 2) / img_w
        y_center = ((b[1] + b[3]) / 2) / img_h
        width = (b[2] - b[0]) / img_w
        height = (b[3] - b[1]) / img_h
        lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    return lines

for xml_file in input_ann.glob("*.xml"):
    label_file = output_labels / (xml_file.stem + '.txt')
    with open(label_file, 'w') as f:
        f.write('\n'.join(convert_xml_to_yolo(xml_file)))


## Split Dataset into Train/Val

In [4]:
import shutil
from sklearn.model_selection import train_test_split
import os

images_path = '../face-mask-data/images'
labels_path = '../face-mask-data/labels'

image_files = [f for f in os.listdir(images_path) if f.endswith('.png') or f.endswith('.jpg')]
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

def copy_data(files, split):
    for f in files:
        base = f.rsplit('.', 1)[0]
        os.makedirs(f'../face-mask-data/{split}/images', exist_ok=True)
        os.makedirs(f'../face-mask-data/{split}/labels', exist_ok=True)
        shutil.copy(os.path.join(images_path, f), f'../face-mask-data/{split}/images/{f}')
        shutil.copy(os.path.join(labels_path, base + '.txt'), f'../face-mask-data/{split}/labels/{base}.txt')

copy_data(train_files, 'train')
copy_data(val_files, 'val')


## Create data.yaml for YOLOv5
train: ../face-mask-data/train/images
val: ../face-mask-data/val/images

nc: 3
names: ['with_mask', 'mask_weared_incorrect', 'without_mask']

In [5]:
with open('data.yaml', 'w') as f:
    f.write("""train: ../face-mask-data/train/images
val: ../face-mask-data/val/images
nc: 3
names: ['with_mask', 'mask_weared_incorrect', 'without_mask']
""")


## Start Training with YOLOv5l

In [6]:
# ✅ Full patch: register safe globals, fix deprecated np.int, patch torch.load
import torch
from models.yolo import Model, DetectionModel, Detect
from models.common import Conv, Bottleneck, SPP, Focus, C3, Concat
from torch.nn import Sequential, Conv2d, BatchNorm2d, ReLU, MaxPool2d, Upsample, ModuleList
import numpy.core.multiarray as np_multiarray

# Register all necessary classes to safely unpickle YOLOv5 weights
torch.serialization.add_safe_globals([
    # YOLO model classes
    Model, DetectionModel, Detect,
    Conv, Bottleneck, SPP, Focus, C3, Concat,

    # Standard PyTorch layers often used internally by YOLO
    Sequential, Conv2d, BatchNorm2d, ReLU, MaxPool2d, Upsample, ModuleList,

    # NumPy object for weight reconstruction
    np_multiarray._reconstruct
])

# 🔧 Patch deprecated usage of np.int in dataloaders.py
dataloader_path = '/kaggle/working/yolov5/utils/dataloaders.py'
with open(dataloader_path, 'r') as f:
    content = f.read()
content = content.replace('np.int', 'int')
with open(dataloader_path, 'w') as f:
    f.write(content)

# 🔧 Patch torch.load in train.py to set weights_only=False
train_path = '/kaggle/working/yolov5/train.py'
with open(train_path, 'r') as f:
    content = f.read()
content = content.replace(
    "ckpt = torch.load(weights, map_location='cpu')",
    "ckpt = torch.load(weights, map_location='cpu', weights_only=False)"
)
with open(train_path, 'w') as f:
    f.write(content)

print("✅ All YOLOv5 classes allowlisted, np.int patched, and torch.load updated.")

✅ All YOLOv5 classes allowlisted, np.int patched, and torch.load updated.


In [7]:
!python train.py --img 640 --batch 16 --epochs 50 --data data.yaml --cfg models/yolov5l.yaml --weights yolov5l.pt --name face-mask-yolov5l

2025-06-10 13:15:39.329764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749561339.779312      82 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749561339.903900      82 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: W&B disabled due to login timeout.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setti

In [8]:
# ✅ Manually load best.pt with weights_only=False to extract state_dict safely
import torch

# Safe load with weights_only=False (required in PyTorch ≥2.6)
model_full = torch.load('runs/train/face-mask-yolov5l/weights/best.pt', map_location='cpu', weights_only=False)

# Extract the actual model if it's inside a dict
if isinstance(model_full, dict) and 'model' in model_full:
    model_full = model_full['model']

torch.save(model_full.state_dict(), 'runs/train/face-mask-yolov5l/weights/best_state_dict.pt')
print("✅ best_state_dict.pt saved from best.pt using safe load.")

✅ best_state_dict.pt saved from best.pt using safe load.


## # ✅ Save safe state_dict version of best.pt after training

In [9]:
from models.yolo import Model
import torch

# ✅ Rebuild model architecture manually (not using attempt_load)
model = Model('models/yolov5l.yaml', ch=3, nc=3)  # ch=3 for RGB, nc=3 for 3 face mask classes
model.load_state_dict(torch.load('runs/train/face-mask-yolov5l/weights/best_state_dict.pt', map_location='cpu'))
model.eval()

print("✅ model.state_dict loaded and ready for inference.")

Overriding model.yaml nc=80 with nc=3

                 from  n    params  module                                  arguments                     
  0                -1  1      7040  models.common.Conv                      [3, 64, 6, 2, 2]              
  1                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  2                -1  3    156928  models.common.C3                        [128, 128, 3]                 
  3                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  4                -1  6   1118208  models.common.C3                        [256, 256, 6]                 
  5                -1  1   1180672  models.common.Conv                      [256, 512, 3, 2]              
  6                -1  9   6433792  models.common.C3                        [512, 512, 9]                 
  7                -1  1   4720640  models.common.Conv                      [512, 1024, 3, 2]            

✅ model.state_dict loaded and ready for inference.
